# 03 — Sealed Test Scoring (3 tiers) + Efficiency Benchmark

**Runs on:** Kaggle GPU T4. **Covers:** TASKS P7.1–P7.4.

**Run this ONCE, only after Phases 2–5 are complete** (temperatures, fusion, scaler, conformal tau
all fitted and saved) AND Phase 6 baselines exist (merged adapter scored, stress set authored and
frozen, external loader written).

This is the only notebook that reads test-tier data: tier 1 UNIFIED-TEST, tier 2 external benchmark,
tier 3 benign stress set. Decision-layer artefacts are applied READ-ONLY — nothing is fitted or
re-thresholded here. Headline metric is TPR@1%FPR with bootstrap CIs.

In [ ]:
REPO_URL = "https://github.com/<YOUR_USER>/<YOUR_REPO>.git"  # TODO
!git clone -q {REPO_URL} slm_shield
%cd slm_shield
# Kaggle/Colab ship a pre-provisioned torch+CUDA. Do NOT `uv sync` here (it would rebuild the
# GPU stack and risk CUDA mismatch). Install the behaviour-critical libs on top of platform torch,
# pinned to the versions declared in pyproject.toml (the same ones your adapters were trained under).
!pip install -q "transformers==4.53.1" "unsloth==2025.7.2" peft trl accelerate bitsandbytes
# If Kaggle's preinstalled versions clash, restart the kernel after install and re-run from here.
from kaggle_secrets import UserSecretsClient
import os
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
ARTIFACT_REPO = "<YOUR_USER>/slm-shield-artifacts"  # TODO

In [ ]:
# --- Pull fitted decision-layer artefacts (temperatures, fusion, scaler, conformal, manifests) ---
from huggingface_hub import snapshot_download
ART = snapshot_download(ARTIFACT_REPO, repo_type="dataset", token=HF_TOKEN)
print("Artefacts at", ART)

In [ ]:
# --- Load backbone + 3 specialists + generalist ---
from src.model_loader import load_model_with_adapters
model, tokenizer = load_model_with_adapters(hf_token=HF_TOKEN, include_generalist=True)

In [ ]:
# --- P7.1: single sealed scoring pass over ALL THREE TEST TIERS ---
# tier 1 = UNIFIED-TEST, tier 2 = external benchmark, tier 3 = benign stress set.
# Decision-layer artefacts are applied READ-ONLY. No fitting, no re-thresholding here.
from src.scoring import bulk_score_split
from src.eval.splits import load_manifests
from src.eval.external import load_external_benchmark
manifests = load_manifests(f"{ART}/manifests")

tier_paths = {}
tier_paths["tier1_unified_test"] = bulk_score_split(
    model, tokenizer, manifests, "UNIFIED-TEST",
    out_path="/kaggle/working/scores_tier1.parquet", resume=True)
manifests.assert_test_distinct(tier_paths["tier1_unified_test"])  # guards duplicated-run bug

external = load_external_benchmark()          # Open-Prompt-Injection, or deepset fallback
tier_paths["tier2_external"] = bulk_score_split(
    model, tokenizer, external, "EXTERNAL",
    out_path="/kaggle/working/scores_tier2.parquet", resume=True)

stress = manifests.load_stress_set("data/benign_stress_set.jsonl")  # all-benign by construction
tier_paths["tier3_stress"] = bulk_score_split(
    model, tokenizer, stress, "STRESS",
    out_path="/kaggle/working/scores_tier3.parquet", resume=True)
print(tier_paths)

In [ ]:
# --- P7.4: efficiency benchmark (median AND p95 latency, VRAM, throughput) ---
# Run in this same session so all numbers are mutually comparable.
from src.eval.metrics import benchmark_runtime
bench = benchmark_runtime(model, tokenizer, manifests.sample("UNIFIED-TEST", n=300, seed=42),
                          modes=["batched", "sequential_no_exit", "sequential_early_exit", "legacy_generate"],
                          report_percentiles=(50, 95))
print(bench)  # median + p95 ms/prompt per mode (benign vs malicious split) + peak VRAM

In [ ]:
# --- P7.2 + P7.3: full matrix (configs a-h x 3 tiers) then bootstrap CIs (CPU) ---
from src.eval.run_eval import run_full_matrix
from src.eval.bootstrap import bootstrap_cis

results = run_full_matrix(tier_scores=tier_paths, artefacts_dir=ART, runtime_bench=bench,
                          out_dir="/kaggle/working/results")
# Headline metric is TPR@1%FPR; accuracy is appendix-only.
print(results.summary())

# 1,000-resample 95% CIs on F1 and TPR@1%FPR — no GPU, resamples saved predictions only.
cis = bootstrap_cis(tier_scores=tier_paths, artefacts_dir=ART, n_resamples=1000,
                    metrics=("f1", "tpr_at_1pct_fpr"), out_dir="/kaggle/working/results")
print(cis.summary())   # states whether specialist-vs-baseline and fusion-vs-OR gaps exceed CIs

from huggingface_hub import HfApi
HfApi(token=HF_TOKEN).upload_folder(folder_path="/kaggle/working/results",
                                    repo_id=ARTIFACT_REPO, repo_type="dataset",
                                    path_in_repo="results")